In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# =========================
# IndoBERTweet + BiLSTM Model
# =========================
class IndoBERTweetBiLSTM(nn.Module):
    def __init__(self, bert_model="indolem/indobertweet-base-uncased", hidden_size=128, num_classes=2, dropout=0.3):
        super(IndoBERTweetBiLSTM, self).__init__()
        self.bert = AutoModel.from_pretrained(bert_model)
        
        # BiLSTM layer
        self.lstm = nn.LSTM(
            input_size=self.bert.config.hidden_size,   # 768 for IndoBERTweet base
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # Fully connected classifier
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 because BiLSTM is bidirectional

    def forward(self, input_ids, attention_mask):
        # BERT embeddings
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # shape: (batch, seq_len, hidden_size)

        # LSTM over sequence
        lstm_out, _ = self.lstm(sequence_output)  # shape: (batch, seq_len, hidden_size*2)

        # Pooling: take last hidden state (or use mean pooling)
        lstm_out = lstm_out[:, -1, :]  # shape: (batch, hidden_size*2)

        # Dropout + classification
        out = self.dropout(lstm_out)
        logits = self.fc(out)  # shape: (batch, num_classes)
        return logits


# =========================
# Example usage
# =========================
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")

# Example batch of texts
texts = [
    "Bosku ayo depo 100k langsung g4c0r 🎰🔥", 
    "Hari ini cerah sekali, enak jalan-jalan."
]

encodings = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

# Initialize model
model = IndoBERTweetBiLSTM(num_classes=2)

# Forward pass
with torch.no_grad():
    logits = model(encodings["input_ids"], encodings["attention_mask"])
    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

print("Predictions:", preds)  # 0 = non-judi, 1 = judi
print("Probabilities:", probs)


In [ ]:
from transformers import TFAutoModel
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Flatten, Dense
from tensorflow.keras.models import Model
import pandas as pd
import tensorflow as tf

# Load your data
df = pd.read_json('fetched_data_final_dedup.json')
X = df['text'].tolist()
y = df['label'].astype(int).tolist()

# Load IndoBERTweet
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")
bert_model = TFAutoModel.from_pretrained("indolem/indobertweet-base-uncased")

max_length = 128
# Tokenize input texts
encodings = tokenizer(X, truncation=True, padding=True, max_length=max_length, return_tensors='tf')

input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']
labels = tf.convert_to_tensor(y)


# Input layer (token IDs)
input_ids = Input(shape=(128,), dtype='int32', name='input_ids')
attention_mask = Input(shape=(128,), dtype='int32', name='attention_mask')

# Get hidden states from IndoBERTweet
bert_output = bert_model(input_ids, attention_mask=attention_mask)
last_hidden_state = bert_output.last_hidden_state  # shape: (None, 128, 768)

# BiLSTM
x = Bidirectional(LSTM(64, return_sequences=True))(last_hidden_state)  # (None, 128, 128)

# Dropout
x = Dropout(0.3)(x)

# Flatten
x = Flatten()(x)  # (None, 16384)

# Dense output
output = Dense(1, activation='sigmoid')(x)

# Build model
model = Model(inputs=[input_ids, attention_mask], outputs=output)

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()
